# Previsão de Rotatividade de Funcionários (Attrition)

*Comparação de regressão logística, random forest, XGBoost e rede neural sob desbalanceamento de classes*

---

Autor: Wellington Moreira

Contato: https://www.linkedin.com/in/wellington-moreira-santos/

E-mail: wsantos08@hotmail.com

## Introdução

Empresas enfrentam custos elevados de contratação e treinamento quando funcionários deixam a organização sem aviso estratégico. Neste projeto, construo um modelo de classificação para prever a rotatividade (attrition) de funcionários a partir do dataset IBM HR Analytics Employee Attrition, disponibilizado no Kaggle, utilizado originalmente no estudo de caso de Recursos Humanos do curso Ciência de Dados para Empresas e Negócios (IAExpert Academy).

O material original do curso treina três modelos (regressão logística, random forest e rede neural) avaliando apenas por acurácia, sem fixar `random_state` ou usar `stratify` na divisão treino/teste, e sem tratar o desbalanceamento das classes, que neste dataset gira em torno de 16% de funcionários que saíram contra 84% que permaneceram. Isso compromete a reprodutibilidade dos resultados e mascara o desempenho real do modelo na classe minoritária, que é a que mais interessa ao negócio.

Neste projeto, expando o escopo original em três frentes. Primeiro, trato o desbalanceamento de classes de forma explícita, comparando cada modelo com e sem ponderação de classes. Segundo, adiciono XGBoost como quarto modelo e uso validação cruzada estratificada para tornar a comparação entre os quatro modelos mais robusta que uma única divisão treino/teste. Terceiro, interpreto os fatores mais relevantes para a previsão cruzando coeficientes da regressão logística com a importância de variáveis de random forest e XGBoost, verificando se os modelos convergem para os mesmos fatores de risco.

As principais ferramentas utilizadas são pandas, numpy, scikit-learn, xgboost, tensorflow/keras, matplotlib e seaborn.

O notebook está organizado nas seguintes seções:

1. Imports e Configurações
2. Coleta e Inspeção dos Dados
3. Análise Exploratória
4. Tratamento e Limpeza dos Dados
5. Preparação para o Modelo
6. Treinamento dos Modelos
7. Avaliação dos Modelos
8. Conclusão e Considerações Finais
9. Referências

## 1. Imports e Configurações

In [3]:
!pip install xgboost tensorflow -q

### 1.1 Importações

In [4]:
import warnings
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
    roc_auc_score, roc_curve, precision_recall_curve, average_precision_score
)

from xgboost import XGBClassifier
import tensorflow as tf

import session_info

### 1.2 Configurações Globais

In [5]:
warnings.filterwarnings('ignore')

RANDOM_STATE = 42

plt.rcParams.update({
    'figure.facecolor': '#0f172a',
    'axes.facecolor':   '#1e293b',
    'axes.edgecolor':   '#334155',
    'axes.labelcolor':  '#94a3b8',
    'xtick.color':      '#94a3b8',
    'ytick.color':      '#94a3b8',
    'text.color':       '#f1f5f9',
    'grid.color':       '#334155',
    'grid.linestyle':   '--',
    'grid.alpha':       0.5,
})

session_info.show(dependencies=False)

## 2. Coleta e Inspeção dos Dados

### 2.1 Carregamento

In [7]:
try:
    employee_df = pd.read_csv('WA_Fn-UseC_-HR-Employee-Attrition.csv')
    print('- DADOS CARREGADOS -')
except FileNotFoundError: print('Erro ao carregar arquivo, verifique o caminho.')
except Exception as e: print(f'Erro: {str(e)} ')

- DADOS CARREGADOS -


### 2.2 Visão Geral

In [8]:
# primeiros registros
employee_df.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


In [9]:
# dimensoes
employee_df.shape

(1470, 35)

In [10]:
# tipo de dados
employee_df.dtypes

Age                         int64
Attrition                     str
BusinessTravel                str
DailyRate                   int64
Department                    str
DistanceFromHome            int64
Education                   int64
EducationField                str
EmployeeCount               int64
EmployeeNumber              int64
EnvironmentSatisfaction     int64
Gender                        str
HourlyRate                  int64
JobInvolvement              int64
JobLevel                    int64
JobRole                       str
JobSatisfaction             int64
MaritalStatus                 str
MonthlyIncome               int64
MonthlyRate                 int64
NumCompaniesWorked          int64
Over18                        str
OverTime                      str
PercentSalaryHike           int64
PerformanceRating           int64
RelationshipSatisfaction    int64
StandardHours               int64
StockOptionLevel            int64
TotalWorkingYears           int64
TrainingTimesL

In [12]:
# nulos 
employee_df.isnull().sum()

Age                         0
Attrition                   0
BusinessTravel              0
DailyRate                   0
Department                  0
DistanceFromHome            0
Education                   0
EducationField              0
EmployeeCount               0
EmployeeNumber              0
EnvironmentSatisfaction     0
Gender                      0
HourlyRate                  0
JobInvolvement              0
JobLevel                    0
JobRole                     0
JobSatisfaction             0
MaritalStatus               0
MonthlyIncome               0
MonthlyRate                 0
NumCompaniesWorked          0
Over18                      0
OverTime                    0
PercentSalaryHike           0
PerformanceRating           0
RelationshipSatisfaction    0
StandardHours               0
StockOptionLevel            0
TotalWorkingYears           0
TrainingTimesLastYear       0
WorkLifeBalance             0
YearsAtCompany              0
YearsInCurrentRole          0
YearsSince

### 2.3 Estatísticas Descritivas

In [14]:
employee_df.describe().T

,count,mean,std,min,25%,50%,75%,max
Age,1470.0,36.923810,9.135373,18.0,30.00,36.0,43.00,60.0
DailyRate,1470.0,802.485714,403.509100,102.0,465.00,802.0,1157.00,1499.0
DistanceFromHome,1470.0,9.192517,8.106864,1.0,2.00,7.0,14.00,29.0
Education,1470.0,2.912925,1.024165,1.0,2.00,3.0,4.00,5.0
EmployeeCount,1470.0,1.000000,0.000000,1.0,1.00,1.0,1.00,1.0
EmployeeNumber,1470.0,1024.865306,602.024335,1.0,491.25,1020.5,1555.75,2068.0
EnvironmentSatisfaction,1470.0,2.721769,1.093082,1.0,2.00,3.0,4.00,4.0
HourlyRate,1470.0,65.891156,20.329428,30.0,48.00,66.0,83.75,100.0
JobInvolvement,1470.0,2.729932,0.711561,1.0,2.00,3.0,3.00,4.0
JobLevel,1470.0,2.063946,1.106940,1.0,1.00,2.0,3.00,5.0


In [15]:
# distribuição do attr alvo
employee_df['Attrition'].value_counts(normalize=True) * 100

Attrition
No     83.877551
Yes    16.122449
Name: proportion, dtype: float64